## Now constructing the folds file used while finetuning the model

In [21]:
import os
import json

In [22]:
os.environ['http_proxy']="http://proxy61.iitd.ac.in:3128"
os.environ['https_proxy']="http://proxy61.iitd.ac.in:3128"

In [23]:
split_number = "without_nan_with_space_remv_run_3"

min_tables = 10
min_rows = 8
max_rows = 12
min_cols = 10
max_cols = 25

In [24]:
corpus_path = "../../dataset/spider/processing/corpus"
corpus_path_split = f"../../dataset/spider/processing/corpus_split/{split_number}_mint_{min_tables}_minr_{min_rows}_maxr_{max_rows}_minc_{min_cols}_maxc_{max_cols}"
corpus_coverate_split = f"../../dataset/spider/processing/corpus_coverage/{split_number}_mint_{min_tables}_minr_{min_rows}_maxr_{max_rows}_minc_{min_cols}_maxc_{max_cols}"

In [25]:
# checking if the split folder exist in that path

if not os.path.exists(corpus_path_split):
    os.makedirs(corpus_path_split)
    print("folder created...")
else:
    print("folder already exist...")

folder created...


## Reading the databases

In [26]:
files_corpus = []

# checking the number of files in query_tables path
if os.path.exists(corpus_path) and os.path.isdir(corpus_path):
    files_corpus = [f for f in os.listdir(corpus_path) if os.path.isfile(os.path.join(corpus_path, f))]
    for d in files_corpus:
        print(d)
else:
    print(f"Folder not found at {corpus_path}")


re_tables-party_people.json
re_tables-architecture.json
re_tables-chinook_1.json
re_tables-phone_market.json
re_tables-flight_company.json
re_tables-swimming.json
re_tables-protein_institute.json
re_tables-school_finance.json
re_tables-department_management.json
re_tables-orchestra.json
re_tables-journal_committee.json
re_tables-real_estate_properties.json
re_tables-driving_school.json
re_tables-county_public_safety.json
re_tables-ship_1.json
re_tables-employee_hire_evaluation.json
re_tables-local_govt_mdm.json
re_tables-dorm_1.json
re_tables-election.json
re_tables-riding_club.json
re_tables-behavior_monitoring.json
re_tables-student_transcripts_tracking.json
re_tables-performance_attendance.json
re_tables-soccer_2.json
re_tables-match_season.json
re_tables-music_1.json
re_tables-culture_company.json
re_tables-gymnast.json
re_tables-party_host.json
re_tables-imdb.json
re_tables-car_1.json
re_tables-sports_competition.json
re_tables-tracking_software_problems.json
re_tables-course_teac

## Reading the tables and creating the corpus with splits

In [27]:
import random

def get_subtables(table, columns, min_tables, min_row, max_row, min_col, max_col, seed=42):

    # setting the random seed for reproducibility
    # random.seed(seed)

    total_col = len(columns)
    total_row = len(table)

    column_cover_track = [0]*total_col ## to track the coverage of the columns
    subtables = []
    
    total_sub_tables = min_tables
    ## if the number of rows in the tables are zero then number of sub-tables shuld not be too much
    if total_row <= min_row:
        total_sub_tables = min(min_tables,max(2, total_col-min_col))

    while True:
        ## Now select the number of columns that need to be selected
        total_col_selected = random.randint(min_col, max_col)
        total_col_selected = min(total_col_selected, total_col)
        # print("total columns selected", total_col_selected)

        column_list = [i for i in range(total_col)]
        column_list_selected = random.sample(column_list, total_col_selected)

        ## Selecting the columns for the subtables
        subtables_columns = []
        for i in column_list_selected:
            subtables_columns.append(columns[i])

        ## Now select the number of rows that need to be selected
        total_row_selected = random.randint(min_row, max_row)
        total_row_selected = min(total_row_selected, total_row)

        ## Selecting the rows for the subtables
        sampled_table_rows = []
        for i in table:
            tmp_row = []
            for j in column_list_selected:
                tmp_row.append(i[j])
            sampled_table_rows.append(tmp_row)
        
        ## now selecting the rows
        row_list = [i for i in range(total_row)]
        row_list_selected = random.sample(row_list, total_row_selected)

        subtables_rows = []
        for i in row_list_selected:
            subtables_rows.append(sampled_table_rows[i])

        subtables.append([subtables_columns, subtables_rows])

        for i in column_list_selected:
            column_cover_track[i] = 1

        if sum(column_cover_track) == total_col and len(subtables) >= total_sub_tables:
            break
    
    return subtables

In [28]:
# ## transforming the tables into the correct format

# augmented_data = {}

# for k in table_content.keys():
#     print(k, end=" : ")

#     table = table_content[k]["data"]
#     title = table_content[k]["title"]
    
#     print("No. of rows:", len(table), ": No. of columns:", len(title))

#     table_list = []
#     augmented_data[k] = get_subtables(table, title, min_tables, min_rows, max_rows, min_cols, max_cols)

In [29]:
def clean_table_nan(table):
    temp_table = []

    cnt = 0
    for t in table:
        if "nan" in t:
            cnt+=1
        else:
            temp_table.append(t)
    
    print(len(table), cnt)
    return temp_table

def replace_table_nan(table):
    temp_table = []

    for t in table:
        if "nan" in t:
            tmp_list = []
            for val in t:
                if val=="nan":
                    tmp_list.append("")
                else:
                    tmp_list.append(val)
            temp_table.append(tmp_list)
        else:
            temp_table.append(t)
    return temp_table

In [30]:
coverage = {}


# Read the table data from the corpus
if len(files_corpus) != 0:
    for f in files_corpus:
        # getting the name of the file
        name = f.strip().split(".")[0]

        new_file_data = {}
        
        # reading the file
        with open(os.path.join(corpus_path,f), 'r') as fp:
            json_data = json.load(fp)
        
        data_keys = json_data.keys()

        # iterating over all the tables in the file
        for k in data_keys:
            table = json_data[k]["data"]
            
            # table = clean_table_nan(table)
            table = replace_table_nan(table)


            title = json_data[k]["title"]

            augmented_tables = get_subtables(table, title, min_tables, min_rows, max_rows, min_cols, max_cols)
            print(k, " rows:",len(json_data[k]["data"]), ": Cols:",len(title) ,": Total sub-tables:", len(augmented_tables))
            
            for idx, t in enumerate(augmented_tables):
                new_table = json_data[k].copy()

                new_table['title'] = t[0]
                new_table["data"] = t[1]
                new_table["numericColumns"] = []
                new_table["numCols"] = len(t[0])
                new_table["numDataRows"] = len(t[1])
                
                new_file_data[f'{k}__{idx}'] = new_table
        
        # writing this in the new location
        with open(os.path.join(corpus_path_split,f"{name}.json"), "w") as f:
            json.dump(new_file_data, f, indent=2)

print("Processing completed")

table-party_people-region  rows: 5 : Cols: 6 : Total sub-tables: 2
table-party_people-party  rows: 5 : Cols: 6 : Total sub-tables: 2
table-party_people-member  rows: 15 : Cols: 4 : Total sub-tables: 10
table-party_people-party_events  rows: 8 : Cols: 4 : Total sub-tables: 2
table-architecture-architect  rows: 5 : Cols: 4 : Total sub-tables: 2
table-architecture-bridge  rows: 15 : Cols: 6 : Total sub-tables: 10
table-architecture-mill  rows: 6 : Cols: 7 : Total sub-tables: 2
table-chinook_1-Album  rows: 347 : Cols: 3 : Total sub-tables: 10
table-chinook_1-Artist  rows: 275 : Cols: 2 : Total sub-tables: 10
table-chinook_1-Customer  rows: 59 : Cols: 13 : Total sub-tables: 10
table-chinook_1-Employee  rows: 8 : Cols: 15 : Total sub-tables: 5
table-chinook_1-Genre  rows: 25 : Cols: 2 : Total sub-tables: 10
table-chinook_1-Invoice  rows: 412 : Cols: 9 : Total sub-tables: 10
table-chinook_1-InvoiceLine  rows: 2240 : Cols: 5 : Total sub-tables: 10
table-chinook_1-MediaType  rows: 5 : Cols: 2 :